In [1]:
import os
import pandas as pd

data_path = os.path.join("..", "data", "mimic-demo", "icu")

chartevents = pd.read_csv(os.path.join(data_path, "chartevents.csv.gz"))
d_items = pd.read_csv(os.path.join(data_path, "d_items.csv.gz"))
icustays = pd.read_csv(os.path.join(data_path, "icustays.csv.gz"))

print("Data loaded successfully")
print("chartevents rows:", len(chartevents))
print("d_items rows:", len(d_items))
print("icustays rows:", len(icustays))

Data loaded successfully
chartevents rows: 668862
d_items rows: 4014
icustays rows: 140


In [2]:
chart = chartevents.merge(d_items, on="itemid", how="left")

print("Chart merged successfully")
print("chart rows:", len(chart))

chart[["stay_id", "itemid", "label", "charttime", "valuenum"]].head()

Chart merged successfully
chart rows: 668862


,stay_id,itemid,label,charttime,valuenum
0,32604416,225054,Anti Embolic Device Status,2132-12-16 00:00:00,NaN
1,32604416,223769,O2 Saturation Pulseoxymetry Alarm - High,2132-12-16 00:00:00,100.0
2,32604416,223956,Temporary Pacemaker Mode,2132-12-16 00:00:00,NaN
3,32604416,224866,Temporary Atrial Capture,2132-12-16 00:00:00,NaN
4,32604416,227341,History of falling (within 3 mnths),2132-12-16 00:00:00,0.0


In [3]:
vitals = chart[
    chart["label"].isin([
        "Heart Rate",
        "Respiratory Rate",
        "Arterial Blood Pressure systolic",
        "O2 saturation pulseoxymetry"
    ])
].copy()

vitals["charttime"] = pd.to_datetime(vitals["charttime"])

print("Vitals filtered:", len(vitals))
print(vitals["label"].value_counts())

Vitals filtered: 46891
label
Respiratory Rate                    13913
Heart Rate                          13913
O2 saturation pulseoxymetry         13540
Arterial Blood Pressure systolic     5525
Name: count, dtype: int64


In [4]:
stay_ids = vitals["stay_id"].dropna().unique()[:100]

print("Number of selected stay_ids:", len(stay_ids))
print(stay_ids[:10])

Number of selected stay_ids: 100
[32604416 36084484 34629895 32145159 34617352 32506122 39804682 39711498
 35258379 37057036]


In [40]:
def build_feature_row_for_stay(stay_id, vitals):
    patient_data = vitals[vitals["stay_id"] == stay_id].copy()

    if patient_data.empty:
        return None

    ref_time = patient_data["charttime"].max()
    window_start = ref_time - pd.Timedelta(hours=6)

    window_data = patient_data[
        (patient_data["charttime"] >= window_start) &
        (patient_data["charttime"] <= ref_time)
    ].copy()

    if window_data.empty:
        return None

    pivot_data = window_data.pivot_table(
        index="charttime",
        columns="label",
        values="valuenum",
        aggfunc="mean"
    ).sort_index()

    pivot_data = pivot_data.ffill()

    if pivot_data.empty or len(pivot_data) < 2:
        return None

    current = pivot_data.iloc[-1]
    first = pivot_data.iloc[0]
    trend = current - first

    # ===== Current values =====
    HR_current = current.get("Heart Rate", None)
    RR_current = current.get("Respiratory Rate", None)
    SpO2_current = current.get("O2 saturation pulseoxymetry", None)
    SBP_current = current.get("Arterial Blood Pressure systolic", None)

    # ===== Trend values =====
    HR_trend = trend.get("Heart Rate", None)
    RR_trend = trend.get("Respiratory Rate", None)
    SpO2_trend = trend.get("O2 saturation pulseoxymetry", None)
    SBP_trend = trend.get("Arterial Blood Pressure systolic", None)

    # ===== Shock Index =====
    shock_index = None
    if HR_current is not None and SBP_current is not None and SBP_current != 0:
        shock_index = HR_current / SBP_current

    # ===== Main reasons =====
    reason = []

    # 🚨 Hard rule: High respiratory rate = Critical
    if RR_current is not None and RR_current >= 28:
        reason.append("Severely high respiratory rate")

    # RR عالي (مهم جدا)
    if current.get("Respiratory Rate", 0) >= 25:
        reason.append("High respiratory rate")

# HR عالي
    if current.get("Heart Rate", 0) >= 100:
        reason.append("High heart rate")

# SpO2 منخفض
    if current.get("O2 saturation pulseoxymetry", 100) <= 93:
        reason.append("Low oxygen saturation")

    if HR_trend is not None and HR_trend >= 10:
        reason.append("Rising heart rate")

    if RR_trend is not None and RR_trend >= 8:
        reason.append("Rising respiratory rate")

    if SpO2_trend is not None and SpO2_trend <= -3:
        reason.append("Falling oxygen saturation")

    if shock_index is not None and shock_index >= 0.9:
        reason.append("High shock index")

    # ===== Early hidden reasons =====
    early_hidden_reasons = []

    if HR_trend is not None and HR_trend >= 10:
        early_hidden_reasons.append("Rising heart rate")

    if RR_trend is not None and RR_trend >= 5:
        early_hidden_reasons.append("Rising respiratory rate")

    if SpO2_trend is not None and SpO2_trend <= -2:
        early_hidden_reasons.append("Falling oxygen saturation")

    if shock_index is not None and shock_index >= 0.8:
        early_hidden_reasons.append("Rising shock index")

   # ===== Attention tier =====
    attention_tier = "Stable"

    # ===== Critical type =====
    critical_type = None

    if attention_tier == "Critical Now":
        
        if RR_current is not None and RR_current >= 28:
            critical_type = "Respiratory Critical"
        
        elif shock_index is not None and shock_index >= 0.9:
            critical_type = "Shock Critical"
        
        elif SpO2_current is not None and SpO2_current <= 92:
            critical_type = "Oxygen Critical"

    # 🔥 Hard rules = Critical
    if (
        (RR_current is not None and RR_current >= 28) or
        (shock_index is not None and shock_index >= 0.9) or
        (SpO2_current is not None and SpO2_current <= 92)
    ):
        attention_tier = "Critical Now"

    # Otherwise use softer logic
    elif len(reason) >= 2:
        attention_tier = "Watch Closely"

    elif len(reason) == 1:
        attention_tier = "Deceptively Stable"

        critical_type = None

    if attention_tier == "Critical Now":

     if RR_current is not None and RR_current >= 28:
        critical_type = "Respiratory Critical"

    elif shock_index is not None and shock_index >= 0.9:
        critical_type = "Shock Critical"

    elif SpO2_current is not None and SpO2_current <= 92:
        critical_type = "Oxygen Critical"

    # ===== Hidden tier =====
    hidden_tier = "No Early Hidden Risk"
    if len(early_hidden_reasons) >= 2:
        hidden_tier = "Deceptively Stable"

    feature_row = {
        "stay_id": stay_id,

        "HR_current": HR_current,
        "RR_current": RR_current,
        "SpO2_current": SpO2_current,
        "SBP_current": SBP_current,

        "HR_trend": HR_trend,
        "RR_trend": RR_trend,
        "SpO2_trend": SpO2_trend,
        "SBP_trend": SBP_trend,

        "shock_index": shock_index,

        "num_reasons": len(reason),
        "num_hidden_reasons": len(early_hidden_reasons),

        "attention_tier": attention_tier,
        "hidden_tier": hidden_tier,
        "critical_type": critical_type,
    }

    return feature_row

In [28]:
test_row = build_feature_row_for_stay(stay_ids[0], vitals)
test_row

{'stay_id': np.int64(32604416),
 'HR_current': np.float64(82.0),
 'RR_current': np.float64(25.0),
 'SpO2_current': np.float64(94.0),
 'SBP_current': None,
 'HR_trend': np.float64(20.0),
 'RR_trend': np.float64(10.0),
 'SpO2_trend': np.float64(-4.0),
 'SBP_trend': None,
 'shock_index': None,
 'num_reasons': 4,
 'num_hidden_reasons': 3,
 'attention_tier': 'Watch Closely',
 'hidden_tier': 'Deceptively Stable'}

In [41]:
all_features = []

for stay_id in stay_ids:
    row = build_feature_row_for_stay(stay_id, vitals)
    if row is not None:
        all_features.append(row)

features_df = pd.DataFrame(all_features)

print("Rows collected before cleaning:", len(features_df))
features_df.head()

Rows collected before cleaning: 100


,stay_id,HR_current,RR_current,SpO2_current,SBP_current,HR_trend,RR_trend,SpO2_trend,SBP_trend,shock_index,num_reasons,num_hidden_reasons,attention_tier,hidden_tier,critical_type
0,32604416,82.0,25.0,94.0,NaN,20.0,10.0,-4.0,NaN,NaN,4,3,Watch Closely,Deceptively Stable,None
1,36084484,90.0,14.0,95.0,100.0,16.0,-5.0,-1.0,-22.0,0.9,2,2,Critical Now,Deceptively Stable,None
2,34629895,84.0,20.0,94.0,NaN,-13.0,-5.0,-2.0,NaN,NaN,0,1,Stable,No Early Hidden Risk,None
3,32145159,96.0,21.0,93.0,NaN,8.0,-2.0,-1.0,NaN,NaN,1,0,Deceptively Stable,No Early Hidden Risk,None
4,34617352,36.0,0.0,90.0,NaN,-66.0,-18.0,0.0,NaN,NaN,1,0,Critical Now,No Early Hidden Risk,None


In [42]:
clean_features_df = features_df[
    (features_df["HR_current"].notna()) &
    (features_df["RR_current"].notna()) &
    (features_df["SpO2_current"].notna()) &
    (features_df["RR_current"] > 0)
].copy()

print("Final dataset size:", len(clean_features_df))

print("\nAttention tier distribution:")
print(clean_features_df["attention_tier"].value_counts())

print("\nHidden tier distribution:")
print(clean_features_df["hidden_tier"].value_counts())

clean_features_df.head()

Final dataset size: 95

Attention tier distribution:
attention_tier
Stable                42
Deceptively Stable    21
Critical Now          20
Watch Closely         12
Name: count, dtype: int64

Hidden tier distribution:
hidden_tier
No Early Hidden Risk    83
Deceptively Stable      12
Name: count, dtype: int64


,stay_id,HR_current,RR_current,SpO2_current,SBP_current,HR_trend,RR_trend,SpO2_trend,SBP_trend,shock_index,num_reasons,num_hidden_reasons,attention_tier,hidden_tier,critical_type
0,32604416,82.0,25.0,94.0,NaN,20.0,10.0,-4.0,NaN,NaN,4,3,Watch Closely,Deceptively Stable,None
1,36084484,90.0,14.0,95.0,100.0,16.0,-5.0,-1.0,-22.0,0.9,2,2,Critical Now,Deceptively Stable,None
2,34629895,84.0,20.0,94.0,NaN,-13.0,-5.0,-2.0,NaN,NaN,0,1,Stable,No Early Hidden Risk,None
3,32145159,96.0,21.0,93.0,NaN,8.0,-2.0,-1.0,NaN,NaN,1,0,Deceptively Stable,No Early Hidden Risk,None
5,32506122,62.0,13.0,97.0,NaN,-9.0,-4.0,-2.0,NaN,NaN,0,1,Stable,No Early Hidden Risk,None


In [31]:
# نشوف الحالات الخطيرة
critical_cases = clean_features_df[
    clean_features_df["attention_tier"] == "Critical Now"
]

critical_cases[[
    "stay_id",
    "HR_current",
    "RR_current",
    "SpO2_current",
    "shock_index",
    "num_reasons"
]]

,stay_id,HR_current,RR_current,SpO2_current,shock_index,num_reasons
1,36084484,90.0,14.0,95.0,0.900000,2
12,39268883,81.0,30.0,97.0,NaN,2
16,30864406,91.0,30.0,97.0,NaN,4
18,38507547,100.0,24.0,98.0,0.990099,2
19,34600477,92.0,28.0,96.0,NaN,2
48,31494479,104.0,24.0,97.0,1.238095,4
57,35889503,116.0,40.0,98.0,NaN,3
64,36871784,70.0,32.0,82.0,NaN,4
65,31316840,86.0,10.0,98.0,1.592593,1
70,30458995,112.0,22.0,97.0,0.941176,2


In [32]:
for i, row in critical_cases.iterrows():
    print("----")
    print("Stay ID:", row["stay_id"])
    print("HR:", row["HR_current"], "| RR:", row["RR_current"], "| SpO2:", row["SpO2_current"])
    print("Shock Index:", row["shock_index"])
    print("Num reasons:", row["num_reasons"])

----
Stay ID: 36084484
HR: 90.0 | RR: 14.0 | SpO2: 95.0
Shock Index: 0.9
Num reasons: 2
----
Stay ID: 39268883
HR: 81.0 | RR: 30.0 | SpO2: 97.0
Shock Index: nan
Num reasons: 2
----
Stay ID: 30864406
HR: 91.0 | RR: 30.0 | SpO2: 97.0
Shock Index: nan
Num reasons: 4
----
Stay ID: 38507547
HR: 100.0 | RR: 24.0 | SpO2: 98.0
Shock Index: 0.9900990099009901
Num reasons: 2
----
Stay ID: 34600477
HR: 92.0 | RR: 28.0 | SpO2: 96.0
Shock Index: nan
Num reasons: 2
----
Stay ID: 31494479
HR: 104.0 | RR: 24.0 | SpO2: 97.0
Shock Index: 1.2380952380952381
Num reasons: 4
----
Stay ID: 35889503
HR: 116.0 | RR: 40.0 | SpO2: 98.0
Shock Index: nan
Num reasons: 3
----
Stay ID: 36871784
HR: 70.0 | RR: 32.0 | SpO2: 82.0
Shock Index: nan
Num reasons: 4
----
Stay ID: 31316840
HR: 86.0 | RR: 10.0 | SpO2: 98.0
Shock Index: 1.5925925925925926
Num reasons: 1
----
Stay ID: 30458995
HR: 112.0 | RR: 22.0 | SpO2: 97.0
Shock Index: 0.9411764705882353
Num reasons: 2
----
Stay ID: 35026312
HR: 125.0 | RR: 33.0 | SpO2: 100.

In [38]:
critical_cases = clean_features_df[
    clean_features_df["attention_tier"] == "Critical Now"
]

deceptive_cases = clean_features_df[
    clean_features_df["attention_tier"] == "Deceptively Stable"
]

print("Critical cases:", len(critical_cases))
print("Deceptively stable cases:", len(deceptive_cases))

print("\nAverage values by tier:")
print(
    clean_features_df.groupby("attention_tier")[
        ["HR_current", "RR_current", "SpO2_current", "HR_trend", "RR_trend", "SpO2_trend"]
    ].mean()
)

Critical cases: 20
Deceptively stable cases: 21

Average values by tier:
                    HR_current  RR_current  SpO2_current   HR_trend  RR_trend  \
attention_tier                                                                  
Critical Now         92.900000   25.550000     94.650000   2.000000  2.600000   
Deceptively Stable   93.714286   20.238095     96.333333   4.947368 -3.000000   
Stable               77.119048   17.404762     97.238095  -5.268293 -2.619048   
Watch Closely        99.416667   19.250000     95.583333  13.166667  2.333333   

                    SpO2_trend  
attention_tier                  
Critical Now         -1.210526  
Deceptively Stable   -0.619048  
Stable                0.666667  
Watch Closely        -1.916667  


In [43]:
print("\nCritical type distribution:")
print(clean_features_df["critical_type"].value_counts())


Critical type distribution:
critical_type
Respiratory Critical    10
Name: count, dtype: int64
